

---

# 📘 LeetCode 2752: Customers with Maximum Number of Transactions on Consecutive Days

---

## ❓ Question

We want to find all `customer_id` who made the **maximum number of transactions on consecutive days**.  

- Each row in the `Transactions` table contains information about transactions with unique `(customer_id, transaction_date)` along with the corresponding `amount`.  
- Return all `customer_id` with the maximum number of consecutive transactions.  
- Order the result table by `customer_id` in ascending order.  

---

## 📊 Sample Data

### Transactions Table

| transaction_id | customer_id | transaction_date | amount |
|----------------|-------------|------------------|--------|
| 1              | 101         | 2023-05-01       | 100    |
| 2              | 101         | 2023-05-02       | 150    |
| 3              | 101         | 2023-05-03       | 200    |
| 4              | 102         | 2023-05-01       | 50     |
| 5              | 102         | 2023-05-03       | 100    |
| 6              | 102         | 2023-05-04       | 200    |
| 7              | 105         | 2023-05-01       | 100    |
| 8              | 105         | 2023-05-02       | 150    |
| 9              | 105         | 2023-05-03       | 200    |

---

### Expected Output

| customer_id |
|-------------|
| 101         |
| 105         |

---

## 🏗️ Schema Definition

```python
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("transaction_date", DateType(), False),
    StructField("amount", IntegerType(), False)
])
```

---

## 📥 Data Preparation

```python
import datetime

transactions_data = [
    (1, 101, datetime.date(2023, 5, 1), 100),
    (2, 101, datetime.date(2023, 5, 2), 150),
    (3, 101, datetime.date(2023, 5, 3), 200),
    (4, 102, datetime.date(2023, 5, 1), 50),
    (5, 102, datetime.date(2023, 5, 3), 100),
    (6, 102, datetime.date(2023, 5, 4), 200),
    (7, 105, datetime.date(2023, 5, 1), 100),
    (8, 105, datetime.date(2023, 5, 2), 150),
    (9, 105, datetime.date(2023, 5, 3), 200)
]
```

---

## 🗂️ Create DataFrame

```python
transactions_df = spark.createDataFrame(transactions_data, schema=transactions_schema)
transactions_df.show()
```

---

## 👁️ Register as SQL View

```python
transactions_df.createOrReplaceTempView("Transactions")
```

---


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

transactions_schema = StructType([
    StructField("transaction_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("transaction_date", DateType(), False),
    StructField("amount", IntegerType(), False)
])
import datetime

transactions_data = [
    (1, 101, datetime.date(2023, 5, 1), 100),
    (2, 101, datetime.date(2023, 5, 2), 150),
    (3, 101, datetime.date(2023, 5, 3), 200),
    (4, 102, datetime.date(2023, 5, 1), 50),
    (5, 102, datetime.date(2023, 5, 3), 100),
    (6, 102, datetime.date(2023, 5, 4), 200),
    (7, 105, datetime.date(2023, 5, 1), 100),
    (8, 105, datetime.date(2023, 5, 2), 150),
    (9, 105, datetime.date(2023, 5, 3), 200)
]
transactions_df = spark.createDataFrame(transactions_data, schema=transactions_schema)
transactions_df.show()
transactions_df.createOrReplaceTempView("Transactions")


In [0]:
%sql
WITH cte AS (
		SELECT coalesce(date_diff(transaction_date, lag(transaction_date) OVER (
						PARTITION BY customer_id ORDER BY transaction_date
						)), 1) AS diff,
			sum(amount) OVER (PARTITION BY customer_id) AS total_earning_per_customer,
			*
		FROM Transactions
		)

SELECT DISTINCT customer_id
FROM cte
WHERE customer_id NOT IN (
		SELECT DISTINCT customer_id
		FROM cte
		WHERE diff > 1
		)
	AND total_earning_per_customer = (
		SELECT max(total_earning_per_customer)
		FROM cte
		)
